# Single Model Evaluation

Quick benchmark for testing your model against the baseline.

**Usage:**
1. Run the Setup cells
2. Edit the Configuration cell to specify your model
3. Run all remaining cells
4. Copy the markdown output to `docs/BENCHMARK.md`

## Table of Contents
1. [Setup](#1-setup)
2. [Configuration](#2-configuration)
3. [Evaluation](#3-evaluation)
4. [Comparison with Baseline](#4-comparison-with-baseline)
5. [Export to BENCHMARK.md](#5-export-to-benchmarkmd)

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
# Import shared evaluation modules
from src.evaluation import (
    load_dataset,
    apply_preprocessing,
    create_fuzzy_post_processor,
    # Formatting
    format_table_md,
    format_accuracy,
    format_latency,
    format_language_report_md,
    format_intent_report_md,
    format_entity_report_md,
    # Confusion matrix
    compute_confusion_matrix,
    plot_confusion_matrix,
)
from src.evaluation.evaluators import (
    evaluate_language_detectors,
    evaluate_intent_classifiers,
    evaluate_entity_extractors,
)
from IPython.display import Markdown, display

print("Imports successful!")

In [ ]:
# Load and preprocess dataset
DATASET_PATH = project_root / "datasets" / "augmented" / "test.csv"
raw_data = load_dataset(str(DATASET_PATH))
data = apply_preprocessing(raw_data, verbose=True)

print(f"\nDataset: {len(data)} samples")

---
## 2. Configuration

**Edit the cell below to specify your model.**

Uncomment ONE of the three options (Intent, Entity, or Language) and configure your model.

In [ ]:
# ============================================================================
# CONFIGURATION - EDIT THIS CELL
# ============================================================================

# Choose ONE option by uncommenting it:

# -----------------------------------------------------------------------------
# Option 1: Intent Classifier
# -----------------------------------------------------------------------------
from src.nlp.intent import SpacyIntentClassifier
my_model = ("SpaCy", SpacyIntentClassifier())
eval_type = "intent"

# -----------------------------------------------------------------------------
# Option 2: Entity Extractor
# -----------------------------------------------------------------------------
# from src.nlp.entity import RegexEntityExtractor
# my_model = ("Regex", RegexEntityExtractor())
# eval_type = "entity"
# use_fuzzy = True  # Set to True to test with fuzzy post-processing

# -----------------------------------------------------------------------------
# Option 3: Language Detector
# -----------------------------------------------------------------------------
# from src.nlp.language import LangdetectLanguageDetector
# my_model = ("Langdetect", LangdetectLanguageDetector())
# eval_type = "language"

# ============================================================================

print(f"Model: {my_model[0]}")
print(f"Type: {eval_type}")

---
## 3. Evaluation

Run your model against the test dataset and display results.

In [ ]:
# Run evaluation based on model type
print(f"Evaluating {my_model[0]}...\n")

if eval_type == "intent":
    results, preds = evaluate_intent_classifiers(
        [my_model], data, return_predictions=True
    )
    result = results[my_model[0]]
    y_true, y_pred = preds[my_model[0]]
    labels = ["TRIP", "NOT_TRIP", "UNKNOWN"]
    
    # Display sklearn-style report
    display(Markdown(format_intent_report_md(my_model[0], result)))

elif eval_type == "entity":
    fuzzy_post = create_fuzzy_post_processor() if use_fuzzy else None
    results, preds = evaluate_entity_extractors(
        [my_model], data, fuzzy_post=fuzzy_post, normalize_fuzzy=use_fuzzy, return_predictions=True
    )
    # Key includes " + Fuzzy" suffix if fuzzy was used
    result_key = f"{my_model[0]} + Fuzzy" if fuzzy_post else my_model[0]
    result = results[result_key]
    entity_preds = preds[result_key]
    
    # Display sklearn-style report
    display(Markdown(format_entity_report_md(result_key, result)))
    
    # For entity, we have departure and destination separately
    y_true = entity_preds.departure_true
    y_pred = entity_preds.departure_pred
    labels = sorted(set(y_true) | set(y_pred))

elif eval_type == "language":
    results, preds = evaluate_language_detectors(
        [my_model], data, return_predictions=True
    )
    result = results[my_model[0]]
    y_true, y_pred = preds[my_model[0]]
    labels = ["FRENCH", "ENGLISH", "UNKNOWN"]
    
    # Display sklearn-style report
    display(Markdown(format_language_report_md(my_model[0], result)))

print(f"\n{'='*50}")
print(f"Accuracy: {result.accuracy:.1%}")
print(f"Latency: {result.avg_latency_ms:.1f}ms")

In [ ]:
# Plot confusion matrix
import matplotlib.pyplot as plt

matrix, cm_labels = compute_confusion_matrix(y_true, y_pred, labels=labels)
fig = plot_confusion_matrix(
    matrix, cm_labels, 
    title=f"{eval_type.title()}: {my_model[0]}", 
    normalize=True
)
plt.show()

---
## 4. Comparison with Baseline

Compare your model against the current best models.

In [ ]:
# Baseline accuracies (from full_evaluation.ipynb)
# Update these values after running full_evaluation.ipynb
BASELINE = {
    "intent": {
        "Regex": 0.658,
        "SpaCy": 0.791,
        "CamemBERT": 0.780,
    },
    "entity": {
        "Regex": 0.465,
        "SpaCy": 0.302,
        "CamemBERT": 0.276,
        "Regex + Fuzzy": 0.571,
        "SpaCy + Fuzzy": 0.342,
        "CamemBERT + Fuzzy": 0.312,
    },
    "language": {
        "Regex": 0.675,
        "Langdetect": 0.779,
    },
}

# Compare
baseline = BASELINE[eval_type]
my_acc = result.accuracy

print(f"{'Model':<20} {'Accuracy':>10} {'Delta':>10}")
print("=" * 42)

# Your model first
print(f"{my_model[0] + ' (yours)':<20} {my_acc:>10.1%} {'--':>10}")
print()

# Baseline models
for name, acc in sorted(baseline.items(), key=lambda x: -x[1]):
    delta = my_acc - acc
    print(f"{name:<20} {acc:>10.1%} {delta:>+10.1%}")

---
## 5. Export to BENCHMARK.md

Copy the markdown below to `docs/BENCHMARK.md`.

In [ ]:
# Generate markdown for BENCHMARK.md
print("=" * 60)
print("COPY THE MARKDOWN BELOW TO docs/BENCHMARK.md")
print("=" * 60)
print()

if eval_type == "intent":
    print(format_intent_report_md(my_model[0], result))
elif eval_type == "entity":
    result_key = f"{my_model[0]} + Fuzzy" if use_fuzzy else my_model[0]
    print(format_entity_report_md(result_key, result))
elif eval_type == "language":
    print(format_language_report_md(my_model[0], result))

print()
print("=" * 60)